In [3]:
pip install sqlalchemy pymysql

     ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
      --------------------------------------- 0.0/2.2 MB 1.4 MB/s eta 0:00:02
      --------------------------------------- 0.0/2.2 MB 1.4 MB/s eta 0:00:02
      --------------------------------------- 0.0/2.2 MB 281.8 kB/s eta 0:00:08
     - -------------------------------------- 0.1/2.2 MB 393.8 kB/s eta 0:00:06
     - -------------------------------------- 0.1/2.2 MB 438.1 kB/s eta 0:00:05
     -- ------------------------------------- 0.1/2.2 MB 364.4 kB/s eta 0:00:06
     --- ------------------------------------ 0.2/2.2 MB 557.9 kB/s eta 0:00:04
     ---- ----------------------------------- 0.2/2.2 MB 655.6 kB/s eta 0:00:03
     ---- ----------------------------------- 0.2/2.2 MB 655.6 kB/s eta 0:00:03
     ---- ----------------------------------- 0.2/2.2 MB 655.6 kB/s eta 0:00:03
     ---- ----------------------------------- 0.2/2.2 MB 655.6 kB/s eta 0:00:03
     -------- ------------------------------- 0.5/2.2


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sqlalchemy import create_engine,text
from sqlalchemy.engine import URL

# SySQL connection details
username = "root"
password = "---------"
host = "localhost"
port = 3306
database = "shopease"

# Safely construct the connection URL
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database
)

# Create engine
engine = create_engine(connection_url)

# Test connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE()"))
    print("Connected database:", result.scalar())


Connected database: shopease


In [8]:
# Verify that the tables are empty

from sqlalchemy import text

with engine.connect() as connection:

    tables = [
        "customers",
        "products",
        "orders",
        "order_items",
        "payments"
    ]

    for table in tables:
        result = connection.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )

        print(table, ":", result.scalar())

customers : 0
products : 0
orders : 0
order_items : 0
payments : 0


In [10]:
import pandas as pd 

# Read customers CSV
customers_df = pd.read_csv("../data/raw/customer.csv")

# Convert signup_date into datetime
customers_df["signup_date"] = pd.to_datetime(
    customers_df["signup_date"]
)

# Check the dataset
print(customers_df.shape)
print(customers_df.head())



(5000, 9)
  customer_id first_name last_name  gender  age       city country  \
0      C00001       Emma     Moore  Female   44  Abu Dhabi     UAE   
1      C00002        Ali    Martin    Male   41  Abu Dhabi     UAE   
2      C00003   Mohammed   Johnson  Female   33      Dubai     UAE   
3      C00004       John    Rahman    Male   46  Abu Dhabi     UAE   
4      C00005       Emma     Ahmed  Female   62  Abu Dhabi     UAE   

  signup_date                            email  
0  2025-12-29        emma.moore00001@gmail.com  
1  2026-03-16        ali.martin00002@gmail.com  
2  2026-06-17  mohammed.johnson00003@gmail.com  
3  2024-07-22       john.rahman00004@gmail.com  
4  2026-06-26        emma.ahmed00005@gmail.com  


# import customer csv to sql 

In [11]:
customers_df.to_sql(
    name="customers",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Customers imported successfully")

Customers imported successfully


In [12]:
with engine.connect() as connection:

    result = connection.execute(
        text("SELECT COUNT(*) FROM customers")
    )

    print("Customers in MySQL:", result.scalar())

Customers in MySQL: 5000


# import product csv to sql 

In [13]:
# Read products CSV
products_df = pd.read_csv("../data/raw/products.csv")

# Inspect the dataset 
print("Shape : ",products_df.shape)
print(products_df.head())
print(products_df.dtypes)

Shape :  (500, 7)
  product_id        product_name     category sub_category    price    cost  \
0     P00001  Skincare Product 1       Beauty     Skincare   494.89  310.74   
1     P00002   Laptops Product 2  Electronics      Laptops   591.51  323.53   
2     P00003  Skincare Product 3       Beauty     Skincare   632.53  400.65   
3     P00004  Skincare Product 4       Beauty     Skincare  1176.05  692.66   
4     P00005      Bags Product 5      Fashion         Bags   806.78  499.14   

   profit_per_unit  
0           184.15  
1           267.98  
2           231.88  
3           483.39  
4           307.64  
product_id          object
product_name        object
category            object
sub_category        object
price              float64
cost               float64
profit_per_unit    float64
dtype: object


In [16]:
# insert products into sql 

products_df.to_sql(
    name="products",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Products imported successfully!")

Products imported successfully!


# import the order csv to sql


In [17]:
orders_df = pd.read_csv("../data/raw/orders.csv")

print("Shape:", orders_df.shape)
print(orders_df.head())
print(orders_df.dtypes)

Shape: (20000, 5)
  order_id customer_id order_status orders_date shipping_city
0   O00001      C02846    Delivered  2025-11-28         Dubai
1   O00002      C00980    Delivered  2025-07-02         Dubai
2   O00003      C03007    Delivered  2024-10-29       Sharjah
3   O00004      C04220    Delivered  2026-02-24       Sharjah
4   O00005      C04939    Delivered  2025-07-16       Sharjah
order_id         object
customer_id      object
order_status     object
orders_date      object
shipping_city    object
dtype: object


In [19]:
# change the date format 
orders_df["orders_date"] = pd.to_datetime(
    orders_df["orders_date"]
)

print(orders_df.dtypes)


order_id                 object
customer_id              object
order_status             object
orders_date      datetime64[ns]
shipping_city            object
dtype: object


In [21]:
# Import orders into MySQL
orders_df.to_sql(
    name="orders",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Orders imported successfully!")

Orders imported successfully!


In [22]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM orders")
    )
    print("Orders in MySQL:", result.scalar())

Orders in MySQL: 20000


# import order item csv to sql 

In [23]:
order_items_df = pd.read_csv("../data/raw/order_items.csv")

print("Shape:", order_items_df.shape)
print(order_items_df.head())
print(order_items_df.dtypes)

Shape: (49925, 9)
  order_item_id order_id product_id  quantity  unit_price  discount  \
0      OI000001   O00001     P00302         3      752.24         5   
1      OI000002   O00001     P00437         1     1608.19         5   
2      OI000003   O00002     P00188         4      405.26        10   
3      OI000004   O00002     P00446         3      328.41         0   
4      OI000005   O00002     P00392         5     1258.14         0   

   gross_amount  discount_amount  total_amount  
0       2256.72           112.84       2143.88  
1       1608.19            80.41       1527.78  
2       1621.04           162.10       1458.94  
3        985.23             0.00        985.23  
4       6290.70             0.00       6290.70  
order_item_id       object
order_id            object
product_id          object
quantity             int64
unit_price         float64
discount             int64
gross_amount       float64
discount_amount    float64
total_amount       float64
dtype: object


In [24]:
print("Missing values:")
print(order_items_df.isnull().sum())

print("\nDuplicate rows:")
print(order_items_df.duplicated().sum())

Missing values:
order_item_id      0
order_id           0
product_id         0
quantity           0
unit_price         0
discount           0
gross_amount       0
discount_amount    0
total_amount       0
dtype: int64

Duplicate rows:
0


In [27]:
# Import into MySQL
order_items_df.to_sql(
    name="order_items",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Order items imported successfully!")

Order items imported successfully!


In [28]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM order_items")
    )
    print("Order items in MySQL:", result.scalar())

Order items in MySQL: 49925


# import the payment csv to the sql

In [29]:
payments_df = pd.read_csv("../data/raw/payments.csv")

print("Shape:", payments_df.shape)
print(payments_df.head())
print(payments_df.dtypes)

Shape: (20000, 6)
  payment_id order_id payment_date    payment_method payment_status  \
0  PAY000001   O00001   2025-11-28    Digital Wallet      Completed   
1  PAY000002   O00002   2025-07-02  Cash on Delivery      Completed   
2  PAY000003   O00003   2024-10-29    Digital Wallet      Completed   
3  PAY000004   O00004   2026-02-24       Credit Card      Completed   
4  PAY000005   O00005   2025-07-16       Credit Card      Completed   

   payment_amount  
0         3671.66  
1         8734.87  
2         5666.10  
3         3742.10  
4          987.32  
payment_id         object
order_id           object
payment_date       object
payment_method     object
payment_status     object
payment_amount    float64
dtype: object


In [30]:
payments_df["payment_date"] = pd.to_datetime(
    payments_df["payment_date"]
)

print(payments_df.dtypes)

payment_id                object
order_id                  object
payment_date      datetime64[ns]
payment_method            object
payment_status            object
payment_amount           float64
dtype: object


In [31]:
payments_df.to_sql(
    name="payments",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Payments imported successfully!")

Payments imported successfully!


In [32]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM payments")
    )
    print("Payments in MySQL:", result.scalar())

Payments in MySQL: 20000
